# Workshop Notebook: Molecular Representations & Regression Models

In this notebook, we will compare **two ligand-based representations** for predicting cytotoxicity of ruthenium(II) metal complexes:

1. **Representation A:** **RDKit molecular descriptors** (computed from individual ligand SMILES)
2. **Representation B:** **Fingerprints** (bit-vector representations of the ligands)

For the fingerprint-based representation, we will simply sum up the vectors for each individual ligand (element by element), for the molecular descriptor representation, we will concatenate the descriptor vectors for each ligand into a single long vector representing the entire complex (summing would likely destroy the signals encoded in the individual ligand descriptors).

We will evaluate both representations under two splitting strategies:
- **Random split** (default splitting strategy, but can overestimate performance),
- **DOI split** (harder, but closer to practical application: tests how well the model *generalizes across papers*).

> **Note on hyperparameters:** The model hyperparameters used below were selected in an extensive hyperparameter search performed separately (not covered in this workshop). Here we focus on **data splits, representations, and interpretation**.

Let's start by downloading all the necessary files.


In [ ]:
! wget https://raw.githubusercontent.com/chimie-paristech-CTM/PSL_notebooks/main/cytotoxicity_metal_complexes_application/lib.zip
! wget https://raw.githubusercontent.com/chimie-paristech-CTM/PSL_notebooks/main/cytotoxicity_metal_complexes_application/ruthenium_complexes_dataset.csv

!unzip lib.zip -d lib

Next, we will set up the environment.

In [ ]:
# ============================================================
# Colab setup (recommended): RDKit via apt + pip extras
# Run this once per new Colab runtime
# ============================================================
import sys, subprocess

def _run(cmd):
    subprocess.check_call(cmd)

def pip_install(pkgs):
    _run([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# System install for RDKit (works reliably on Colab)
_run(["bash", "-lc", "apt-get -qq update"])
_run(["bash", "-lc", "apt-get -qq install -y python3-rdkit rdkit-data"])

# Python packages
pip_install(["pip", "setuptools", "wheel"])
pip_install(["numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn", "tqdm", "mols2grid", "ipywidgets"])

from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
# ============================================================
# (Optional) Mount Google Drive and set the project directory
# This makes relative paths behave like on your local machine.
# ============================================================
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# 👉 Edit this if you want to run from Drive.
# If you cloned a repo, you can set PROJECT_DIR to that folder instead.
PROJECT_DIR = os.getcwd()  # e.g. "/content/drive/MyDrive/Qlife_winter_school"

os.chdir('/content/drive/MyDrive/Colab Notebooks/digital_workshop2')
print("Working directory:", os.getcwd())


In [ ]:
# Helper: soft checks (warn instead of stopping the notebook)
def soft_assert(condition, message=""):
    """Soft check: prints a warning instead of stopping the notebook."""
    if condition:
        print(f"✅ {message}".strip())
        return True
    else:
        print(f"⚠️ {message}".strip())
        return False


In [ ]:
#utility functions : prepare the data 
from lib.utils import prepare_df_morgan, prepare_df_rdkit
from lib.utils import average_duplicates, calc_desc

#utility functions : CV and results 
from lib.utils import plot_cv_results
from lib.utils import df_split, get_indices, get_indices_doi, get_indices_scaff
from lib.utils import cross_validation

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Visualization
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 20})

from itertools import *

## Representation A: RDKit descriptors

We compute a vector of RDKit molecular descriptors for each ligand, then build a complex-level feature vector by combining the three ligands.


In [ ]:
metals = pd.read_csv("ruthenium_complexes_dataset.csv", dtype={'L1': str, 'L2': str, 'L3': str})

In [ ]:
metals_rdkit = prepare_df_rdkit(metals)

In [ ]:
metals_rdkit = average_duplicates(metals_rdkit, 'Ligands_Dict', 'pIC50')

## Computing molecular descriptor representation from RDKit

In [ ]:
metals_rdkit = calc_desc(metals_rdkit)

### Your turn
Check that *metals_rdkit* contains a *Desc1*, *Desc2* and *Desc3* column, i.e., one for every ligand, and check how many descriptors have been computed for each.


In [ ]:
# YOUR CODE HERE


## Splitting the dataset

We will use two splitting strategies:

- **Random split:** a standard train/test split. The splitting indices are given by the function **get_indices**.
- **DOI split:** split by publication (DOI) to probe generalization across papers. The splitting indices are given by the function **get_indices_doi**.


# Training the model

The parameters of the random forest were chosen by running Bayesian Optimisation on the hyperparameters search space with the descriptor representation. 

In [ ]:
rf = RandomForestRegressor(max_depth=26, n_estimators=390, min_samples_leaf=1, max_features=0.5, random_state=42)

## Standard Splitting

#### On the whole dataset

In [ ]:
metals_rdkit_copy = metals_rdkit.copy()

X = np.array(metals_rdkit_copy['Descriptors'].values.tolist())
y = np.array(metals_rdkit_copy['pIC50'].values.tolist())

indices_metals = get_indices(metals_rdkit_copy, CV=10, sizes=(0.9, 0.1))

In [ ]:
y_data, y_predictions = cross_validation(metals_rdkit_copy, indices_metals, X, y, rf, descriptors=True)

In [ ]:
plot_cv_results(y_data, y_predictions, log=True)

## DOI Splitting

We split the dataset based on the DOI.

#### Training the model on the whole dataset

In [ ]:
metals_doi_copy = metals_rdkit.copy()

#We drop any row where the descriptors contains a NaN value, as the model will not take any NaN value.
for index, array in enumerate(metals_doi_copy['Descriptors']):
    if np.isnan(array).any():
        metals_doi_copy.drop(index, inplace=True)
metals_doi_copy.reset_index(drop=True, inplace=True)

X_doi = np.array(metals_doi_copy['Descriptors'].values.tolist())
y_doi = np.array(metals_doi_copy['pIC50'].values.tolist())

indices_metals_doi = get_indices_doi(metals_doi_copy, CV=10, sizes=(0.9, 0.1))

In [ ]:
y_data_doi, y_predictions_doi = cross_validation(metals_doi_copy, indices_metals_doi, X_doi, y_doi, rf, descriptors=True)

In [ ]:
plot_cv_results(y_data_doi, y_predictions_doi, log=True)

## Representation B: Fingerprints

Here we represent each ligand using a **fingerprint** (bit vector) and then build a complex-level representation by combining the three ligand fingerprints.


In [ ]:
metals = pd.read_csv("ruthenium_complexes_dataset.csv", dtype={'L1': str, 'L2': str, 'L3': str})

In [ ]:
metals_morgan_fp = prepare_df_morgan(metals, 2, 1024)
metals_rdkit_fp = prepare_df_rdkit(metals, nbits=2048)

In [ ]:
metals_morgan_fp = average_duplicates(metals_morgan_fp, 'Ligands_Dict', 'pIC50')

In [ ]:
metals_rdkit_fp = average_duplicates(metals_rdkit_fp, 'Ligands_Dict', 'pIC50')

### Your turn
Check that *metals_rdkit_fp* contains a *RDKIT_1*, *RDKIT_2* and *RDKIT_3* column, i.e., one for every ligand, as well as a *Fingerprint* column, representing the entire complex. Next, check the dimensionality of this final Fingerprint.


In [ ]:
# YOUR CODE HERE

# Training the model

Bayesian Optimisation allowed us to get the best hyperparameters for the random forest model, depending on the Fingerprint used. The RDKit Fingerprint with the best results was for **n=2, bits=2048**.

In [ ]:
rf_rdkit_fp = RandomForestRegressor(max_depth=15, n_estimators=50, min_samples_leaf=1, max_features=0.5, random_state=42)

## Standard Splitting

#### On the whole dataset, RDKit fingerprint

In [ ]:
metals_rdkit_fp_copy = metals_rdkit_fp.copy()
metals_rdkit_fp_copy.reset_index(drop=True, inplace=True)

X = np.array(metals_rdkit_fp_copy['Fingerprint'].values.tolist())
y = np.array(metals_rdkit_fp_copy['pIC50'].values.tolist())

indices_rdkit_fp = get_indices(metals_rdkit_fp_copy, CV=10)

In [ ]:
y_data_rdkit_fp, y_predictions_rdkit_fp = cross_validation(metals_rdkit_fp_copy, indices_rdkit_fp, X, y, rf_rdkit_fp)

In [ ]:
plot_cv_results(y_data_rdkit_fp, y_predictions_rdkit_fp, log=True)

## DOI Splitting

#### On the whole dataset, RDKit fingerprint

In [ ]:
metals_rdkit_fp_doi = metals_rdkit_fp.copy()
metals_rdkit_fp_doi.reset_index(drop=True, inplace=True)

X_rdkit_fp_doi = np.array(metals_rdkit_fp_doi['Fingerprint'].values.tolist())
y_rdkit_fp_doi = np.array(metals_rdkit_fp_doi['pIC50'].values.tolist())

indices_rdkit_fp_doi = get_indices_doi(metals_rdkit_fp_doi, CV=10, sizes=(0.9, 0.1))

In [ ]:
y_data_rdkit_fp_doi, y_predictions_rdkit_fp_doi = cross_validation(metals_rdkit_fp_doi, indices_rdkit_fp_doi, X_rdkit_fp_doi, y_rdkit_fp_doi, rf_rdkit_fp)

In [ ]:
plot_cv_results(y_data_rdkit_fp_doi, y_predictions_rdkit_fp_doi, log=True)

## Compare representations

At this point you have results for:
- descriptors + random split
- descriptors + DOI split
- fingerprints + random split
- fingerprints + DOI split

### Reflection questions
- Which split strategy gives *more optimistic* performance? Why?
- Which representation seems more robust under DOI splitting?
- What are plausible reasons for performance differences between descriptors and fingerprints?


## CHALLENGE: Repeat the analysis on HeLa cells only

To focus on a single experimental context, repeat **both** representation workflows using only the subset of measurements for **HeLa** cells.

### Your turn
1. Filter the dataframe to HeLa only (look for the appropriate column name, e.g. `Cells`).
2. Recompute (or reuse) the representations on that subset.
3. Run **random split** and **DOI split** again.
4. Compare performance to the full dataset.

> Tip: Keep your code modular. If you wrote helper functions to build `X, y`, you can reuse them here by passing a filtered dataframe.


In [ ]:
# YOUR CODE HERE
